In [1]:
## import numpy as np
from pyswarms.single.global_best import GlobalBestPSO
from pyswarms.utils.functions import single_obj as fx
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import warnings
warnings.filterwarnings("ignore")
import nest_asyncio
nest_asyncio.apply()
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
import tensorflow_federated as tff
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall
from imblearn.over_sampling import RandomOverSampler,SMOTE

2023-08-05 04:26:07,624 - numexpr.utils - INFO - NumExpr defaulting to 8 threads.


In [2]:
def input_spec():
    return (
        tf.TensorSpec([None, 28], tf.float64),
        tf.TensorSpec([None, 1], tf.int64)
    )
def build_model():
    
    final_model =  tf.keras.models.Sequential([
        tf.keras.layers.InputLayer(input_shape=(28,)),
        #tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(32, activation='relu'),
        #tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(64, activation='relu'),
        #tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(32, activation='relu'),
        #tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])

    return final_model

def compile_model():
    """
    Compile a given keras model using SparseCategoricalCrossentropy
    loss and the Adam optimizer with set evaluation metrics.
    """
    keras_model=build_model()
    keras_model.compile(
        loss=tf.keras.losses.BinaryCrossentropy(),
        #optimizer=optimizer.optimize(fx,10),
        optimizer=tf.keras.optimizers.Adam(),
        metrics=[BinaryAccuracy()]
    )

    return keras_model


In [65]:
def model_fn():
    """
    Create TFF model from compiled Keras model and a sample batch.
    """

    global_model = compile_model()

    final_keras_model =global_model
    keras_model_clone = tf.keras.models.clone_model(final_keras_model)

    return tff.learning.from_keras_model(keras_model_clone,     input_spec=input_spec(),
        loss=tf.keras.losses.BinaryCrossentropy(),metrics=[BinaryAccuracy(), Precision(), Recall()])

In [3]:
################################################ Without Scaling ######################################
################################################ Smote           ######################################
################################################ SGD             ######################################

#Federated Learning Tutorial¶
#Author: Daniyal Shahrokhian     16/7/2021
import matplotlib.pyplot as plt
%matplotlib notebook
%matplotlib inline

import nest_asyncio
nest_asyncio.apply()

import time

import numpy as np
from pyswarms.single.global_best import GlobalBestPSO
from pyswarms.utils.functions import single_obj as fx
from pyswarms.utils.plotters import plot_surface
from pyswarms.utils.plotters.formatters import Animator, Designer, Mesher
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
import tensorflow_federated as tff
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall
from imblearn.over_sampling import RandomOverSampler,SMOTE
#from sklearn.metrics import f1_score,make_scorer
SEED = 1337
tf.random.set_seed(SEED)
df = pd.read_csv("C:/Users/Dell5570/creditcard.csv")              
# Creating Alice and Bob's splits:
alice_df = df[:len(df.index)//2]
bob_df = df[len(df.index)//2:]
#Federated Learning Approach
#Data Loading

def make_tf_dataset(dataframe,  batch_size=None):
    # Preprocess the data
    X = dataframe.drop(['Time','Amount', 'Class'], axis=1)
    y = dataframe['Class']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,shuffle=True,stratify=y)
    sm = SMOTE(random_state=42)
    X_train, y_train = sm.fit_resample(X_train, y_train)

    balanced_df=X_train.assign(Class=y_train)
 
    y = balanced_df.pop('Class')
    # Dataset creation
    dataset = tf.data.Dataset.from_tensor_slices((balanced_df.values, y.to_frame().values))
    dataset = dataset.shuffle(2048, seed=SEED)
    if batch_size:
        dataset = dataset.batch(batch_size)

    return dataset
EPOCHS = 100
BATCH_SIZE = 64
train_data, val_data = [], []

for client_data in [alice_df, bob_df]:
    train_df, val_df = train_test_split(client_data, test_size=0.1, random_state=SEED)

      # TF Datasets
    train_data.append(make_tf_dataset(train_df, batch_size=BATCH_SIZE))
    val_data.append(make_tf_dataset(val_df, batch_size=1))

In [69]:
trainer = tff.learning.build_federated_averaging_process(
    model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.SGD(learning_rate=0.1,momentum=0.2),
    server_optimizer_fn=lambda: tf.keras.optimizers.SGD(learning_rate=0.1,momentum=0.2)
)

EPOCHS = 100
import time
start = time.time()
state = trainer.initialize()
#state.model.assign_weights_to(model_fn)
train_hist = []
for i in range(EPOCHS):
    state, metrics = trainer.next(state, train_data)
    train_hist.append(metrics)

    print(f"\rRun {i+1}/{EPOCHS}", end="")
end=time.time()
print(f"Runtime of the program is {end - start}")

Run 100/100Runtime of the program is 255.18078446388245


In [70]:
evaluator = tff.learning.build_federated_evaluation(model_fn)
federated_metrics = evaluator(state.model, val_data)
federated_metrics

OrderedDict([('binary_accuracy', 0.9511804),
             ('precision', 0.9204769),
             ('recall', 0.9876907),
             ('loss', 0.17884742)])

In [5]:
X = df.drop(['Time','Amount', 'Class'], axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
sm = SMOTE(random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)


In [72]:
model = compile_model()

# Set HHO parameters

model.count_params()

5153

In [74]:
def get_shape(model):
    weights_layer = model.get_weights()
    shapes = []
    for weights in weights_layer:
        shapes.append(weights.shape)
    return shapes
def set_shape(weights,shapes):
    new_weights = []
    index=0
    for shape in shapes:
        if(len(shape)>1):
            n_nodes = np.prod(shape)+index
        else:
            n_nodes=shape[0]+index
        tmp = np.array(weights[index:n_nodes]).reshape(shape)
        new_weights.append(tmp)
        index=n_nodes
    return new_weights
def evaluate_nn(W, shape,X_train=X_train, Y_train=y_train):
    results = []

    for weights in W:
        model.set_weights(set_shape(weights,shape))
        score = model.evaluate(X_train, Y_train, verbose=0)
        results.append(1-score[1])
    return results
model=compile_model()
shape = get_shape(model)
x_max = 1.0 * np.ones(5153)
x_min = -1.0 * x_max
bounds = (x_min, x_max)
options = {'c1': 0.4, 'c2': 0.6, 'w': 0.4}
optimizer = GlobalBestPSO(n_particles=10, dimensions=5153,
                          options=options, bounds=bounds)
import time
start = time.time()
cost, pos = optimizer.optimize(evaluate_nn, 10, X_train=X_train, Y_train=y_train,shape=shape)
model.set_weights(set_shape(pos,shape))
end=time.time()
print(f"Runtime of the program is {end - start}")

2023-06-14 12:28:36,892 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.4, 'c2': 0.6, 'w': 0.4}
pyswarms.single.global_best: 100%|██████████|10/10, best_cost=0.206
2023-06-14 12:40:29,511 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.20579159259796143, best pos: [ 0.21125187  0.80260775  0.58597718 ...  0.30787018  0.58208634
 -0.40205271]


Runtime of the program is 712.6896438598633


In [75]:
def model_fn():
    """
    Create TFF model from compiled Keras model and a sample batch.
    """

    #global_model = compile_model()

    final_keras_model =model
    keras_model_clone = tf.keras.models.clone_model(final_keras_model)

    return tff.learning.from_keras_model(keras_model_clone,     input_spec=input_spec(),
        loss=tf.keras.losses.BinaryCrossentropy(),metrics=[BinaryAccuracy(), Precision(), Recall()])

In [82]:
trainer = tff.learning.build_federated_averaging_process(
    model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.0001),
    server_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=1.0)
)

EPOCHS = 100
import time
start = time.time()
state = trainer.initialize()
#state.model.assign_weights_to(model_fn)
train_hist = []
for i in range(EPOCHS):
    state, metrics = trainer.next(state, train_data)
    train_hist.append(metrics)

    print(f"\rRun {i+1}/{EPOCHS}", end="")
end=time.time()
print(f"Runtime of the program is {end - start}")

Run 100/100Runtime of the program is 309.0460407733917


In [83]:
evaluator = tff.learning.build_federated_evaluation(model_fn)
federated_metrics = evaluator(state.model, val_data)
federated_metrics

OrderedDict([('binary_accuracy', 0.9541698),
             ('precision', 0.9559135),
             ('recall', 0.95225745),
             ('loss', 0.48116583)])

In [84]:
from sklearn.metrics import f1_score
def get_shape(model):
    weights_layer = model.get_weights()
    shapes = []
    for weights in weights_layer:
        shapes.append(weights.shape)
    return shapes
def set_shape(weights,shapes):
    new_weights = []
    index=0
    for shape in shapes:
        if(len(shape)>1):
            n_nodes = np.prod(shape)+index
        else:
            n_nodes=shape[0]+index
        tmp = np.array(weights[index:n_nodes]).reshape(shape)
        new_weights.append(tmp)
        index=n_nodes
    return new_weights
def evaluate_model(model, X, y):
    # Evaluate the model on the given data
    y_pred = model.predict(X)
    y_pred = (y_pred > 0.5).astype(int)
    f1 = f1_score(y, y_pred)

    return f1

def objective_function(w,model):
    # Set the weights of the local model
    shape = get_shape(model)
    model.set_weights((set_shape(w,shape)))

    # Evaluate the local model on the local data
    f1_local = evaluate_model(model, X_train, y_train)

    return -f1_local


def normalize(probabilities):
    total = sum(probabilities)
    if total == 0:
        return [1 / len(probabilities)] * len(probabilities)
    return [p / total for p in probabilities]

def aco_fl(objective_function, num_ants, num_iterations, q0, alpha, beta, rho, lb, ub, global_model,min_pheromone=1e-12, max_pheromone=1e6, scale_pheromone=1):
    # Initialize the pheromone matrix and the probability matrix
    pheromone = np.ones_like(lb) * scale_pheromone
    probability = np.ones_like(lb) / len(lb)
    best_position = np.zeros_like(lb)
    best_fitness = objective_function(best_position,  global_model)
    #pheromone = np.ones_like(lb)
    #probability = np.ones_like(lb)/len(lb)
    #best_position = np.zeros_like(lb)
    #best_fitness = objective_function(best_position,  global_model)
    
    # Main loop of the ACO algorithm
    for i in range(num_iterations):
        # Initialize the ant solutions
        print('“---- Iteration {} ----”'.format(i + 1))
        print('“Pheromone:”', pheromone)
        print('“Probability:”', probability)
        print('“Current position:”', best_position)
        solutions = np.zeros((num_ants, len(lb)))
        fitness = np.zeros(num_ants)
        
        # Construct the ant solutions
        for j in range(num_ants):
            current_position = np.zeros_like(lb)
            for k in range(len(lb)):
                # Choose the next hyperparameter using the probability matrix and the pheromone matrix
                if np.random.rand() < q0:
                    next_position = np.argmax(probability)
                else:
                    
                    probabilities = pheromone**alpha * probability**beta
                    probabilities = probabilities/np.sum(probabilities)
                    probabilities = normalize(probabilities)
                    if np.isnan(probabilities).any():
                        probabilities = [1 / len(probabilities)] * len(probabilities)
                    #probabilities = np.divide(probabilities, np.sum(probabilities), out=np.zeros_like(probabilities), where=np.sum(probabilities) != 0)
                    #probabilities =np.nan_to_num(probabilities ,nan=0)
                    #probabilities = np.nan_to_num(probabilities)
                    #print('“Before normalization:”', probabilities)  # Add this print statement
                    
                    #print('“After normalization:”', probabilities)  # Add this print statement
                   # probabilities = normalize(probabilities)
                    next_position = np.random.choice(len(lb), p=probabilities)
                current_position[next_position] = 1
                
            # Evaluate the fitness of the ant solution
            solutions[j] = lb + (np.array(ub)-np.array(lb))*current_position
            fitness[j] = objective_function(solutions[j],  global_model)
            
            # Update the global best solution
            if fitness[j] < best_fitness:
                best_fitness = fitness[j]
                best_position = solutions[j]
        
        # Update the pheromone matrix


        delta_pheromone = np.zeros_like(pheromone)
        for j in range(num_ants):
            for k in range(len(lb)):
                if solutions[j][k] == 1:
                    epsilon = 1e-9
        
                    #delta_pheromone=pd.DataFrame(delta_pheromone)
                    #fitness = fitness.replace([np.inf, -np.inf, -0], 0)
                    delta_pheromone[k] += 1/(fitness[j]+epsilon)

        pheromone = (1-rho)*pheromone + rho*delta_pheromone           
        

        pheromone = np.clip(pheromone, min_pheromone, max_pheromone)

        
        # Update the probability matrix
        probability = pheromone**alpha * probability**beta
        probability = probability/np.sum(probability)


    return best_position, best_fitness


# Define the global model
#global_model = build_model()
#global_model.compile(loss='binary_crossentropy', optimizer=Adam(), metrics=['accuracy'])



q0=0.2
alpha=0.7
beta=0.1
rho=0.2
# Run the ACO algorithm
num_ants = 3
num_iterations = 5
global_model=compile_model()
m = global_model.count_params()
lb = np.full(m, -1)
ub = np.full(m, 1)
import time
start = time.time()
best_position, best_fitness = aco_fl(objective_function, num_ants, num_iterations, q0, alpha, beta, rho, lb, ub,global_model,min_pheromone=1e-12, max_pheromone=1e6, scale_pheromone=1)
end=time.time()
print(f"Runtime of the program is {end - start}")

“---- Iteration 1 ----”
“Pheromone:” [1 1 1 ... 1 1 1]
“Probability:” [0.00019406 0.00019406 0.00019406 ... 0.00019406 0.00019406 0.00019406]
“Current position:” [0 0 0 ... 0 0 0]
“---- Iteration 2 ----”
“Pheromone:” [1.e-12 2.e-01 1.e-12 ... 1.e-12 1.e-12 1.e-12]
“Probability:” [4.08539058e-12 3.32624764e-04 4.08539058e-12 ... 4.08539058e-12
 4.08539058e-12 4.08539058e-12]
“Current position:” [ 1. -1.  1. ...  1.  1. -1.]
“---- Iteration 3 ----”
“Pheromone:” [1.e-12 1.e-12 1.e-12 ... 1.e-12 1.e-12 1.e-12]
“Probability:” [2.90370281e-10 1.79483390e-09 2.90370281e-10 ... 2.90370281e-10
 2.90370281e-10 2.90370281e-10]
“Current position:” [ 1. -1.  1. ...  1.  1. -1.]
“---- Iteration 4 ----”
“Pheromone:” [1.e-12 1.e-12 1.e-12 ... 1.e-12 1.e-12 1.e-12]
“Probability:” [4.30045497e-15 5.15966629e-15 4.30045497e-15 ... 4.30045497e-15
 4.30045497e-15 4.30045497e-15]
“Current position:” [ 1. -1.  1. ...  1.  1. -1.]
“---- Iteration 5 ----”
“Pheromone:” [1.e-12 1.e-12 1.e-12 ... 1.e-12 1.e-12 1.

In [85]:
def model_fn():
    """
    Create TFF model from compiled Keras model and a sample batch.
    """

    #global_model = compile_model()

    final_keras_model =global_model
    keras_model_clone = tf.keras.models.clone_model(final_keras_model)

    return tff.learning.from_keras_model(keras_model_clone,     input_spec=input_spec(),
        loss=tf.keras.losses.BinaryCrossentropy(),metrics=[BinaryAccuracy(), Precision(), Recall()])

In [86]:
trainer = tff.learning.build_federated_averaging_process(
    model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.0001),
    server_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=1.0)
)

EPOCHS = 100
import time
start = time.time()
state = trainer.initialize()
#state.model.assign_weights_to(model_fn)
train_hist = []
for i in range(EPOCHS):
    state, metrics = trainer.next(state, train_data)
    train_hist.append(metrics)

    print(f"\rRun {i+1}/{EPOCHS}", end="")
end=time.time()
print(f"Runtime of the program is {end - start}")

Run 100/100Runtime of the program is 308.62421774864197


In [87]:
evaluator = tff.learning.build_federated_evaluation(model_fn)
federated_metrics = evaluator(state.model, val_data)
federated_metrics

OrderedDict([('binary_accuracy', 0.9674243),
             ('precision', 0.968124),
             ('recall', 0.96667695),
             ('loss', 0.32767397)])

In [96]:
############################### CSO_FL_CCFD ###############################
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
from sklearn.metrics import f1_score
from collections import defaultdict
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE


# Define the objective function
def evaluate_model(model, X, y):
    # Evaluate the model on the given data
    y_pred = model.predict(X)
    y_pred = (y_pred > 0.5).astype(int)
    f1 = f1_score(y, y_pred)

    return f1

def objective_function(w,model):
    # Set the weights of the local model
    shape = get_shape(model)
    model.set_weights((set_shape(w,shape)))

    # Evaluate the local model on the local data
    f1_local = evaluate_model(model, X_train, y_train)

    return -f1_local

# Define the CSO algorithm with federated learning
def cso_fl(objective_function, num_cats, num_iterations, lb, ub,  global_model):
    # Initialize the population of cats with random positions and velocities
    positions = np.random.uniform(lb, ub, size=(num_cats, len(lb)))
    velocities = np.zeros_like(positions)
    best_position = positions[0]
    best_fitness = objective_function(positions[0],  global_model)
    
    # Main loop of the CSO algorithm
    for i in range(num_iterations):
        # Evaluate the fitness of each cat
        fitness = np.array([objective_function(p,  global_model) for p in positions])
        
        # Update the global best position
        index = np.argmin(fitness)
        if fitness[index] < best_fitness:
            best_fitness = fitness[index]
            best_position = positions[index]
        
        # Update the position and velocity of each cat
        for j in range(num_cats):
            # Update the velocity of the cat
            r1 = np.random.random(size=velocities.shape[1])
            r2 = np.random.random(size=velocities.shape[1])
            velocities[j] = velocities[j] + r1*(best_position - positions[j]) + r2*(positions[index] - positions[j])
            
            # Update the position of the cat
            positions[j] = positions[j] + velocities[j]
            
            # Check the boundaries of the position
            positions[j] = np.clip(positions[j], lb, ub)
    
    return best_position, best_fitness



#client_data = [(x_train[i::num_clients], y_train[i::num_clients]) for i in range(num_clients)]
# Define the global model
global_model = compile_model()
#global_model.compile(loss='binary_crossentropy', optimizer=Adam(), metrics=['accuracy'])

# Set the hyperparameter search space

m = global_model.count_params()
lb = np.full(m, -1)
ub = np.full(m, 1)

# Run the CSO algorithm
num_cats = 5
num_iterations = 10
import time
start = time.time()
best_position, best_fitness = cso_fl(objective_function, num_cats, num_iterations, lb, ub,  global_model)
end=time.time()
print(f"Runtime of the program is {end - start}")

Runtime of the program is 243.1384711265564


In [97]:
def model_fn():
    """
    Create TFF model from compiled Keras model and a sample batch.
    """

    #global_model = compile_model()

    final_keras_model =global_model
    keras_model_clone = tf.keras.models.clone_model(final_keras_model)

    return tff.learning.from_keras_model(keras_model_clone,     input_spec=input_spec(),
        loss=tf.keras.losses.BinaryCrossentropy(),metrics=[BinaryAccuracy(), Precision(), Recall()])

In [98]:
trainer = tff.learning.build_federated_averaging_process(
    model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.0001),
    server_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=1.0)
)

EPOCHS = 100
import time
start = time.time()
state = trainer.initialize()
#state.model.assign_weights_to(model_fn)
train_hist = []
for i in range(EPOCHS):
    state, metrics = trainer.next(state, train_data)
    train_hist.append(metrics)

    print(f"\rRun {i+1}/{EPOCHS}", end="")
end=time.time()
print(f"Runtime of the program is {end - start}")

Run 100/100Runtime of the program is 282.24412870407104


In [99]:
evaluator = tff.learning.build_federated_evaluation(model_fn)
federated_metrics = evaluator(state.model, val_data)
federated_metrics

OrderedDict([('binary_accuracy', 0.9587858),
             ('precision', 0.97770756),
             ('recall', 0.93898094),
             ('loss', 0.6286587)])

In [8]:
####################################### HHO_FL_CCFD ####################################################
from sklearn.metrics import f1_score
def evaluate_model(model, X, y):
    # Evaluate the model on the given data
    y_pred = model.predict(X)
    y_pred = (y_pred > 0.5).astype(int)
    f1 = f1_score(y, y_pred)

    return f1

def objective_function(w):
    # Set the weights of the local model
    shape = get_shape(model)
    model.set_weights((set_shape(w,shape)))

    # Evaluate the local model on the local data
    f1_local = evaluate_model(model, X_train, y_train)

    return -f1_local

def hho_federated_learning(f, n, m, alpha, beta, gamma, lb, ub, iter_max):
    """
    Harris Hawks Optimization for Federated Learning

    Parameters:
    f (function): Objective function to optimize
    n (int): Number of Harris hawks
    m (int): Number of weights in each local model
    alpha (float): Harris hawks' step size
    beta (float): Harris hawks' switching probability
    gamma (float): Harris hawks' shrinking coefficient
    lb (ndarray): Lower bounds of the search space
    ub (ndarray): Upper bounds of the search space
    iter_max (int): Maximum number of iterations

    Returns:
    g_best (float): Global best objective value
    x_best (ndarray): Global best solution
    """
    # Initialize Harris hawks
    x = np.random.uniform(lb, ub, (n, m))
    f_x = np.apply_along_axis(f, 1, x)
    g_best = np.min(f_x)
    x_best = x[np.argmin(f_x)]

    # Main loop
    for t in range(iter_max):
        print(t)
        # Calculate fitness values
        fitness = 1 / (1 + f_x)

        # Sort Harris hawks by fitness values
        idx = np.argsort(fitness)[::-1]
        x = x[idx]
        fitness = fitness[idx]

        # Update Harris hawks' positions
        for i in range(n):
            if np.random.rand() < beta:
                # Exploration: move towards the global best solution
                x[i] = x[i] + alpha * (x_best - x[i]) + gamma * np.random.randn(m)
            else:
                # Exploitation: move towards the best solution in the neighborhood
                j = np.random.choice(np.delete(np.arange(n), i))
                x[i] = x[i] + alpha * (x[j] - x[i]) + gamma * np.random.randn(m)

            # Apply bounds to the positions
            x[i] = np.clip(x[i], lb, ub)

        # Evaluate new positions
        f_x = np.apply_along_axis(f, 1, x)
        g_best_new = np.min(f_x)
        x_best_new = x[np.argmin(f_x)]

        # Update global best solution
        if g_best_new < g_best:
            g_best = g_best_new
            x_best = x_best_new

    return g_best, x_best




# Build a neural network model
model = compile_model()

# Set HHO parameters
n = 10
m = model.count_params()
alpha = 0.3
beta = 0.6
gamma = 0.9
lb = np.full(m, -1)
ub = np.full(m, 1)
def get_shape(model):
    weights_layer = model.get_weights()
    shapes = []
    for weights in weights_layer:
        shapes.append(weights.shape)
    return shapes
def set_shape(weights,shapes):
    new_weights = []
    index=0
    for shape in shapes:
        if(len(shape)>1):
            n_nodes = np.prod(shape)+index
        else:
            n_nodes=shape[0]+index
        tmp = np.array(weights[index:n_nodes]).reshape(shape)
        new_weights.append(tmp)
        index=n_nodes
    return new_weights
import time
start = time.time()
hho_federated_learning(objective_function, n, m, alpha, beta, gamma, lb, ub, 10)
end=time.time()
print(f"Runtime of the program is {end - start}")


0
1
2
3
4
5
6
7
8
9
Runtime of the program is 777.9005818367004


In [9]:
def model_fn():
    """
    Create TFF model from compiled Keras model and a sample batch.
    """

    #global_model = compile_model()

    final_keras_model =model
    keras_model_clone = tf.keras.models.clone_model(final_keras_model)

    return tff.learning.from_keras_model(keras_model_clone,     input_spec=input_spec(),
        loss=tf.keras.losses.BinaryCrossentropy(),metrics=[BinaryAccuracy(), Precision(), Recall()])

In [10]:
trainer = tff.learning.build_federated_averaging_process(
    model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.0001),
    server_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=1.0)
)

EPOCHS = 100
import time
start = time.time()
state = trainer.initialize()
#state.model.assign_weights_to(model_fn)
train_hist = []
for i in range(EPOCHS):
    state, metrics = trainer.next(state, train_data)
    train_hist.append(metrics)

    print(f"\rRun {i+1}/{EPOCHS}", end="")
end=time.time()
print(f"Runtime of the program is {end - start}")

Run 100/100Runtime of the program is 293.5143835544586


In [12]:
evaluator = tff.learning.build_federated_evaluation(model_fn)
federated_metrics = evaluator(state.model, val_data)
federated_metrics

OrderedDict([('binary_accuracy', 0.94706994),
             ('precision', 0.96370435),
             ('recall', 0.92913353),
             ('loss', 0.81396997)])

In [121]:
def input_spec():
    return (
        tf.TensorSpec([None, X_train.shape[1]], tf.float64),
        tf.TensorSpec([None, 1], tf.int64)
    )
def build_model(learning_rate, num_filters):
    
    final_model =  tf.keras.models.Sequential([
        tf.keras.layers.InputLayer(input_shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(num_filters, activation='relu'),
        tf.keras.layers.Dense(num_filters*2, activation='relu'),
        #tf.keras.layers.Dense(num_filters, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    
    
    final_model.compile(
        loss=tf.keras.losses.BinaryCrossentropy(),
        #optimizer=optimizer.optimize(fx,10),
        optimizer=tf.keras.optimizers.Adam(),
        metrics=[BinaryAccuracy(), Precision(), Recall()]
    )


    """
    Compile a given keras model using SparseCategoricalCrossentropy
    loss and the Adam optimizer with set evaluation metrics.
    """

   
    return final_model

In [122]:
import heapq
class HeapBasedOptimizer:
    def __init__(self):
        self.performance_heap = []

    def add_model(self, model, performance):
        heapq.heappush(self.performance_heap, (-performance, model))

    def get_best_model(self):
        return heapq.heappop(self.performance_heap)[1]
import time
start = time.time()
# Heap based optimizer
optimizer = HeapBasedOptimizer()

# Define range for hyperparameters
learning_rates = [0.001, 0.0005]
num_filters_values = [32]

# Optimize the CNN model
epochs = 10
for learning_rate in learning_rates:
    for num_filters in num_filters_values:
        model = build_model(learning_rate, num_filters)
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0)
        performance = max(history.history["val_binary_accuracy"])  # You can use other performance metrics if desired
        optimizer.add_model(model, performance)

# Get the best model from the heap
best_model = optimizer.get_best_model()
end=time.time()
print(f"Runtime of the program is {end - start}")

Runtime of the program is 232.8514654636383


In [123]:
def model_fn():
    """
    Create TFF model from compiled Keras model and a sample batch.
    """

   # keras_model = build_model()

    final_keras_model =best_model 
    keras_model_clone = tf.keras.models.clone_model(final_keras_model)

    return tff.learning.from_keras_model(keras_model_clone,     input_spec=input_spec(),
        loss=tf.keras.losses.BinaryCrossentropy(),metrics=[BinaryAccuracy(), Precision(), Recall()])

In [130]:
trainer = tff.learning.build_federated_averaging_process(
    model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.01),
    server_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.01)
)

EPOCHS = 100
import time
start = time.time()
state = trainer.initialize()
#state.model.assign_weights_to(model_fn)
train_hist = []
for i in range(EPOCHS):
    state, metrics = trainer.next(state, train_data)
    train_hist.append(metrics)

    print(f"\rRun {i+1}/{EPOCHS}", end="")
end=time.time()
print(f"Runtime of the program is {end - start}")

Run 100/100Runtime of the program is 334.6377058029175


In [131]:
evaluator = tff.learning.build_federated_evaluation(model_fn)
federated_metrics = evaluator(state.model, val_data)
federated_metrics

OrderedDict([('binary_accuracy', 0.97021586),
             ('precision', 0.96258974),
             ('recall', 0.9784587),
             ('loss', 0.08605242)])

In [ ]:
def get_shape(model):
    weights_layer = model.get_weights()
    shapes = []
    for weights in weights_layer:
        shapes.append(weights.shape)
    return shapes
def set_shape(weights,shapes):
    new_weights = []
    index=0
    for shape in shapes:
        if(len(shape)>1):
            n_nodes = np.prod(shape)+index
        else:
            n_nodes=shape[0]+index
        tmp = np.array(weights[index:n_nodes]).reshape(shape)
        new_weights.append(tmp)
        index=n_nodes
    return new_weights
def evaluate_model(model, X, y):
    # Evaluate the model on the given data
    y_pred = model.predict(X)
    y_pred = (y_pred > 0.5).astype(int)
    f1 = f1_score(y, y_pred)

    return f1

def objective_function(w):
    # Set the weights of the local model
    shape = get_shape(model)
    model.set_weights((set_shape(w,shape)))

    # Evaluate the local model on the local data
    f1_local = evaluate_model(model, X_train, y_train)

    return -f1_local

In [ ]:
########################## FHO_FL_CCFD ###########################
from mealpy.swarm_based.FHO import OriginalFHO
from mealpy import FloatVar, GTO
import numpy as np
from sklearn.metrics import f1_score
model=compile_model()
x_max = 1.0 * np.ones(5153)
x_min = -1.0 * x_max
X = df.drop(['Time','Amount', 'Class'], axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
sm = SMOTE(random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)
problem_dict1 = {
    "fit_func": objective_function,

    "lb": x_min,
    "ub": x_max,
    "minmax": "min",
}

epoch = 10
pop_size = 10

import time
start = time.time()

opt_model = GTO.Matlab101GTO(epoch, pop_size)
best_position, best_fitness = opt_model.solve(problem_dict1)
print(f"Solution: {best_position}, Fitness: {best_fitness}")
end=time.time()
print(f"Runtime of the program is {end - start}")

In [ ]:
def model_fn():
    """
    Create TFF model from compiled Keras model and a sample batch.
    """

   # keras_model = build_model()

    final_keras_model =opt_model
    keras_model_clone = tf.keras.models.clone_model(final_keras_model)

    return tff.learning.from_keras_model(keras_model_clone,     input_spec=input_spec(),
        loss=tf.keras.losses.BinaryCrossentropy(),metrics=[BinaryAccuracy(), Precision(), Recall()])

In [ ]:
trainer = tff.learning.build_federated_averaging_process(
    model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.01),
    server_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.01)
)

EPOCHS = 100
import time
start = time.time()
state = trainer.initialize()
#state.model.assign_weights_to(model_fn)
train_hist = []
for i in range(EPOCHS):
    state, metrics = trainer.next(state, train_data)
    train_hist.append(metrics)

    print(f"\rRun {i+1}/{EPOCHS}", end="")
end=time.time()
print(f"Runtime of the program is {end - start}")

In [ ]:
evaluator = tff.learning.build_federated_evaluation(model_fn)
federated_metrics = evaluator(state.model, val_data)
federated_metrics